In [4]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import HTML, display

# --------------------------------------------------
# Config
# --------------------------------------------------
BASE_DIR = Path(".").resolve()
ACC_MAP_DIR = BASE_DIR / "acc_mappings"

EXCHANGES = ["isx", "cox", "hex", "obx", "stx"]

MAPPING_DIRS = {
    ex: ACC_MAP_DIR / f"acc_mappings_{ex}"
    for ex in EXCHANGES
}

XLSX_DIRS = {
    ex: BASE_DIR.parent / "data" / f"{ex}_xlsx"
    for ex in EXCHANGES
}


In [5]:

# --------------------------------------------------
# Helpers
# --------------------------------------------------
def load_mapping(mapping_path: Path) -> dict:
    with open(mapping_path, "r", encoding="utf-8") as f:
        return json.load(f)

def get_variable_entry(mapping: dict, variable_name: str) -> dict | None:
    for var in mapping.get("variables", []):
        if var.get("variable") == variable_name:
            return var
    return None

def read_sheet_first_col_as_str(xlsx_path: Path, sheet_name: str) -> pd.Series:
    """
    Read the first column (column A) from the given sheet as strings.
    """
    df = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None)
    if df.empty:
        return pd.Series(dtype="object")

    col_a = df.iloc[:, 0]
    col_a = col_a.astype(str).str.strip()
    col_a = col_a[col_a.notna()]
    return col_a

def find_duplicate_exact_matches(col_a: pd.Series, row_label: str) -> int:
    """
    Count exact matches of row_label in column A.
    """
    return (col_a == row_label).sum()


In [6]:

# --------------------------------------------------
# Main scan
# --------------------------------------------------
rows = []
missing_files = []
missing_sheets = []

for ex in EXCHANGES:
    mapping_dir = MAPPING_DIRS[ex]
    xlsx_dir = XLSX_DIRS[ex]

    mapping_files = sorted(mapping_dir.glob("*.json"))
    print(f"{ex.upper()}: found {len(mapping_files)} mapping files")

    for mp in mapping_files:
        ticker = mp.stem                      # e.g. EQNR.OL
        xlsx_path = xlsx_dir / f"{ticker}.xlsx"

        if not xlsx_path.exists():
            missing_files.append({
                "ExchangeFolder": ex,
                "Ticker": ticker,
                "MissingFile": str(xlsx_path)
            })
            continue

        mapping = load_mapping(mp)
        std_entry = get_variable_entry(mapping, "STD")
        if std_entry is None:
            continue

        final_choices = std_entry.get("final_choice", [])
        if not final_choices:
            continue

        # cache sheets so we do not reread the same sheet several times per file
        sheet_cache = {}

        for choice in final_choices:
            sheet_name = choice.get("sheet_name")
            row_label = choice.get("row_label")

            if not sheet_name or not row_label:
                continue

            if sheet_name not in sheet_cache:
                try:
                    sheet_cache[sheet_name] = read_sheet_first_col_as_str(xlsx_path, sheet_name)
                except Exception as e:
                    missing_sheets.append({
                        "ExchangeFolder": ex,
                        "Ticker": ticker,
                        "SheetName": sheet_name,
                        "Error": str(e)
                    })
                    continue

            col_a = sheet_cache[sheet_name]
            n_matches = find_duplicate_exact_matches(col_a, row_label)

            if n_matches > 1:
                rows.append({
                    "ExchangeFolder": ex,
                    "Ticker": ticker,
                    "SheetName": sheet_name,
                    "STD_row_label": row_label,
                    "ExactMatchCount": int(n_matches)
                })

# --------------------------------------------------
# Output
# --------------------------------------------------
dup_df = pd.DataFrame(rows).sort_values(
    ["ExchangeFolder", "Ticker", "STD_row_label"]
).reset_index(drop=True)

print("\nFirms with duplicate exact matches for STD final_choice rows:")

html = dup_df.to_html(index=False)

display(HTML(f"""
<div style="max-height:500px; overflow-y:auto; overflow-x:auto; border:1px solid #ccc;">
    {html}
</div>
"""))

print(f"\nTotal duplicate cases: {len(dup_df)}")
print(f"Unique firms with at least one duplicate STD row: {dup_df['Ticker'].nunique() if not dup_df.empty else 0}")

if missing_sheets:
    missing_sheets_df = pd.DataFrame(missing_sheets)
    print("\nCould not read some sheets:")
    display(missing_sheets_df)

ISX: found 22 mapping files
COX: found 89 mapping files
HEX: found 121 mapping files
OBX: found 148 mapping files
STX: found 264 mapping files

Firms with duplicate exact matches for STD final_choice rows:


ExchangeFolder,Ticker,SheetName,STD_row_label,ExactMatchCount
cox,AAB.CO,Balance Sheet,Mortgage,2
cox,AGATE.CO,Balance Sheet,Credit institutions,2
cox,ATLA.CO,Balance Sheet,Short term bank debt,2
cox,BAVA.CO,Balance Sheet,Debt to credit institutions,2
cox,BO.CO,Balance Sheet,Bank loans,2
cox,BO.CO,Balance Sheet,Loans from Banks,2
cox,CHEMM.CO,Balance Sheet,Bank Loans,2
cox,COLOb.CO,Balance Sheet,Other credit institutions,3
cox,COLUM.CO,Balance Sheet,Debt to credit institutions,2
cox,DEMANT.CO,Balance Sheet,Borrowings,2



Total duplicate cases: 178
Unique firms with at least one duplicate STD row: 164
